In [1]:
import os
import numpy as np
import torch
import rasterio

from rasterio.windows import Window
from rasterio.features import shapes
import geopandas as gpd
from shapely.geometry import shape
from tqdm import tqdm


# ==========================================================
# FILE PATHS & PARAMETERS
# ==========================================================

# Input Diepsloot sub-image
DIEPSLOOT_IMAGE_PATH = (
    "/content/drive/MyDrive/Image data/Testing/Diepsloot.tif"
)

# Optional: Ground truth buildings for Diepsloot (if available for accuracy testing)
DIEPSLOOT_GT_PATH = (
    "/content/drive/MyDrive/Image data/Ground truth digitized data/Diepsloot ground truth.gpkg"
)

# Model weight paths from your training outputs
UNET_WEIGHTS_PATH = (
    "/content/drive/MyDrive/Image data/UNet outputs/models/best_unet.pth"
)

DEEPLAB_WEIGHTS_PATH = (
    "/content/drive/MyDrive/Image data/DeepLab outputs/models/best_model.pth"
)

MASKRCNN_WEIGHTS_PATH = (
    "/content/drive/MyDrive/Image data/MaskRCNN/models/best_maskrcnn.pth"
)


# ==========================================================
# OUTPUT DIRECTORY
# ==========================================================

OUTPUT_DIR = (
    "/content/drive/MyDrive/Image data/Diepsloot_Generalization_Results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ==========================================================
# SLIDING WINDOW SETTINGS
# ==========================================================

TILE_SIZE = 1024
OVERLAP = 0.40

STRIDE = int(TILE_SIZE * (1 - OVERLAP))  # 614 pixels


# ==========================================================
# COMPUTING DEVICE
# ==========================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)

Using device: cuda


# Instance Segmentation Generalization Script (Mask R-CNN)

In [2]:
import os
import numpy as np
import torch
import torchvision
import rasterio
import geopandas as gpd
import cv2

from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

from rasterio.windows import Window
from rasterio.features import shapes
from shapely.geometry import shape
from tqdm import tqdm


# ==========================================================
# CONFIGURATION & PATHS
# ==========================================================

DIEPSLOOT_IMAGE_PATH = (
    "/content/drive/MyDrive/Image data/Testing/Diepsloot.tif"
)

MASKRCNN_WEIGHTS_PATH = (
    "/content/drive/MyDrive/Image data/MaskRCNN/models/best_maskrcnn.pth"
)

OUTPUT_DIR = (
    "/content/drive/MyDrive/Image data/Diepsloot_Generalization_Results"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ==========================================================
# SCALE ALIGNMENT PARAMETERS
# ==========================================================

MELUSI_GSD = 0.0800
DIEPSLOOT_GSD = 0.0230635

SCALE_FACTOR = MELUSI_GSD / DIEPSLOOT_GSD

TARGET_TILE_SIZE = 1024

READ_WINDOW_SIZE = int(
    TARGET_TILE_SIZE * SCALE_FACTOR
)

OVERLAP = 0.40

READ_STRIDE = int(
    READ_WINDOW_SIZE * (1 - OVERLAP)
)

SCORE_THRESHOLD = 0.25
MASK_THRESHOLD = 0.50

print("Using device:", DEVICE)
print(f"Scale factor: {SCALE_FACTOR:.3f}")
print(f"Read window size: {READ_WINDOW_SIZE} pixels")
print(f"Read stride: {READ_STRIDE} pixels")
print(f"Detection score threshold: {SCORE_THRESHOLD}")
print(f"Mask threshold: {MASK_THRESHOLD}")


# ==========================================================
# LOAD MASK R-CNN MODEL
# ==========================================================

model = maskrcnn_resnet50_fpn(
    weights=None
)

num_classes = 2  # Background + building

# Replace bounding box predictor
in_features = (
    model.roi_heads.box_predictor.cls_score.in_features
)

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    num_classes
)

# Replace mask predictor
in_features_mask = (
    model.roi_heads.mask_predictor.conv5_mask.in_channels
)

model.roi_heads.mask_predictor = MaskRCNNPredictor(
    in_features_mask,
    256,
    num_classes
)

# Load trained weights
model.load_state_dict(
    torch.load(
        MASKRCNN_WEIGHTS_PATH,
        map_location=DEVICE
    )
)

model.to(DEVICE)
model.eval()

print("Mask R-CNN model loaded successfully.")


# ==========================================================
# INFERENCE WITH SCALE ALIGNMENT
# ==========================================================

with rasterio.open(DIEPSLOOT_IMAGE_PATH) as src:

    width = src.width
    height = src.height

    profile = src.profile.copy()
    transform = src.transform
    crs = src.crs

    # Initialize full prediction map
    full_prediction = np.zeros(
        (height, width),
        dtype=np.uint8
    )

    print(
        f"Running rescaled Mask R-CNN inference on image size "
        f"({width} x {height})..."
    )

    # Process overlapping windows
    for row in tqdm(range(0, height, READ_STRIDE)):

        for col in range(0, width, READ_STRIDE):

            # Calculate window dimensions
            w_w = min(
                READ_WINDOW_SIZE,
                width - col
            )

            w_h = min(
                READ_WINDOW_SIZE,
                height - row
            )

            if w_w <= 0 or w_h <= 0:
                continue

            window = Window(
                col,
                row,
                w_w,
                w_h
            )

            # Read the three input bands
            # Rasterio uses 1-based band indexing
            rgb_tile = src.read(
                [1, 2, 3],
                window=window
            )

            # Convert from [C, H, W] to [H, W, C]
            tile_hwc = np.transpose(
                rgb_tile,
                (1, 2, 0)
            )

            # Resize tile to match training dimensions
            resized_tile = cv2.resize(
                tile_hwc,
                (TARGET_TILE_SIZE, TARGET_TILE_SIZE),
                interpolation=cv2.INTER_LINEAR
            )

            # Normalize pixel values to 0-1
            tensor_tile = (
                resized_tile.astype(np.float32) / 255.0
            )

            # Convert from [H, W, C] to [C, H, W]
            tensor_tile = np.transpose(
                tensor_tile,
                (2, 0, 1)
            )

            # Convert to PyTorch tensor
            tile_tensor = torch.tensor(
                tensor_tile,
                dtype=torch.float32
            ).to(DEVICE)

            # Run Mask R-CNN prediction
            with torch.no_grad():

                prediction = model([tile_tensor])[0]

            scores = (
                prediction["scores"]
                .cpu()
                .numpy()
            )

            masks = (
                prediction["masks"]
                .cpu()
                .numpy()
            )

            # Filter detections by confidence score
            keep = scores >= SCORE_THRESHOLD

            filtered_masks = masks[keep]

            if len(filtered_masks) > 0:

                # Combine detected instance masks into
                # one binary building mask for this tile
                tile_pred_mask = np.any(
                    filtered_masks[:, 0, :, :] >= MASK_THRESHOLD,
                    axis=0
                ).astype(np.uint8)

                # Resize prediction to native image resolution
                pred_mask_native = cv2.resize(
                    tile_pred_mask,
                    (w_w, w_h),
                    interpolation=cv2.INTER_NEAREST
                )

                # Merge into full prediction map
                full_prediction[
                    row:row + w_h,
                    col:col + w_w
                ] = np.maximum(
                    full_prediction[
                        row:row + w_h,
                        col:col + w_w
                    ],
                    pred_mask_native
                )


# ==========================================================
# SAVE PREDICTION RASTER
# ==========================================================

raster_out = os.path.join(
    OUTPUT_DIR,
    "maskrcnn_diepsloot_prediction.tif"
)

profile.update(
    count=1,
    dtype=rasterio.uint8,
    compress="lzw",
    tiled=True,
    blockxsize=256,
    blockysize=256
)

with rasterio.open(
    raster_out,
    "w",
    **profile
) as dst:

    dst.write(full_prediction, 1)

print(f"Prediction raster saved to: {raster_out}")


# ==========================================================
# CONVERT BUILDING MASK TO POLYGONS
# ==========================================================

polygons = [
    shape(geom)
    for geom, val in shapes(
        full_prediction,
        mask=(full_prediction == 1),
        transform=transform
    )
    if val == 1
]


# ==========================================================
# SAVE BUILDING FOOTPRINTS AS GEOPACKAGE
# ==========================================================

vector_out = os.path.join(
    OUTPUT_DIR,
    "maskrcnn_diepsloot_buildings.gpkg"
)

gdf_pred = gpd.GeoDataFrame(
    {
        "class": ["building"] * len(polygons)
    },
    geometry=polygons,
    crs=crs
)

gdf_pred.to_file(
    vector_out,
    driver="GPKG"
)

print(
    f"Mask R-CNN completed. Extracted {len(polygons)} building "
    f"footprints -> {vector_out}"
)


# ==========================================================
# RELEASE GPU MEMORY
# ==========================================================

del model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Using device: cuda
Scale factor: 3.469
Read window size: 3551 pixels
Read stride: 2130 pixels
Detection score threshold: 0.25
Mask threshold: 0.5
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 137MB/s]


Mask R-CNN model loaded successfully.
Running rescaled Mask R-CNN inference on image size (28087 x 19866)...


100%|██████████| 10/10 [01:14<00:00,  7.41s/it]


Prediction raster saved to: /content/drive/MyDrive/Image data/Diepsloot_Generalization_Results/maskrcnn_diepsloot_prediction.tif
Mask R-CNN completed. Extracted 1105 building footprints -> /content/drive/MyDrive/Image data/Diepsloot_Generalization_Results/maskrcnn_diepsloot_buildings.gpkg


# Generalization Accuracy &amp; Performance Evaluation

In [6]:

import os
import gc
import numpy as np
import geopandas as gpd
import rasterio

from rasterio.features import rasterize
from rasterio.windows import Window
from shapely.geometry import box


def evaluate_model_generalization(
    gt_vector_path,
    pred_raster_path,
    window_size=512
):
    """
    Calculate pixel-level Precision, Recall, F1-score and IoU
    using window-based processing to reduce RAM consumption.
    """

    if not os.path.exists(gt_vector_path):
        print(f"Ground truth file not found: {gt_vector_path}")
        return

    if not os.path.exists(pred_raster_path):
        print(f"Prediction raster not found: {pred_raster_path}")
        return

    # Load and clean ground-truth polygons
    gdf_gt = gpd.read_file(gt_vector_path)

    gdf_gt = gdf_gt[
        gdf_gt.geometry.notna()
        & ~gdf_gt.geometry.is_empty
    ].copy()

    if gdf_gt.empty:
        print("Ground truth contains no valid geometries.")
        return

    # Remove invalid geometries where possible
    invalid = ~gdf_gt.geometry.is_valid
    if invalid.any():
        gdf_gt.loc[invalid, "geometry"] = (
            gdf_gt.loc[invalid, "geometry"].make_valid()
        )

    gdf_gt = gdf_gt[
        gdf_gt.geometry.notna()
        & ~gdf_gt.geometry.is_empty
    ].copy()

    # Open prediction raster without loading it all into RAM
    with rasterio.open(pred_raster_path) as src:

        if src.crs is None:
            raise ValueError("Prediction raster has no CRS.")

        if gdf_gt.crs is None:
            raise ValueError("Ground-truth data has no CRS.")

        # Reproject ground truth only if necessary
        if gdf_gt.crs != src.crs:
            gdf_gt = gdf_gt.to_crs(src.crs)

        # Build spatial index for faster polygon lookup
        spatial_index = gdf_gt.sindex

        # Accumulate confusion-matrix counts
        TP = 0
        FP = 0
        FN = 0

        # Process the raster in small windows
        for row_off in range(0, src.height, window_size):
            for col_off in range(0, src.width, window_size):

                win_width = min(
                    window_size, src.width - col_off
                )
                win_height = min(
                    window_size, src.height - row_off
                )

                window = Window(
                    col_off,
                    row_off,
                    win_width,
                    win_height
                )

                # Read only this section of the prediction raster
                pred_window = src.read(1, window=window)

                # Ignore raster NoData pixels if a mask is available
                valid_pixels = src.read_masks(1, window=window) > 0

                # Find ground-truth polygons near this window
                window_bounds = rasterio.windows.bounds(
                    window, src.transform
                )
                window_polygon = box(*window_bounds)

                candidate_indices = spatial_index.query(
                    window_polygon
                )

                # Rasterize only the relevant polygons
                if len(candidate_indices) > 0:
                    shapes_gt = (
                        (geom, 1)
                        for geom in gdf_gt.geometry.iloc[
                            candidate_indices
                        ]
                        if geom is not None and not geom.is_empty
                    )

                    gt_window = rasterize(
                        shapes=shapes_gt,
                        out_shape=(
                            int(win_height),
                            int(win_width)
                        ),
                        transform=src.window_transform(window),
                        fill=0,
                        dtype=np.uint8
                    )
                else:
                    gt_window = np.zeros(
                        (int(win_height), int(win_width)),
                        dtype=np.uint8
                    )

                # Convert predictions to binary classes:
                # 0 = background, positive values = building
                y_pred = pred_window > 0
                y_true = gt_window == 1

                # Count correctly and incorrectly classified pixels
                TP += np.count_nonzero(
                    y_true & y_pred & valid_pixels
                )
                FP += np.count_nonzero(
                    ~y_true & y_pred & valid_pixels
                )
                FN += np.count_nonzero(
                    y_true & ~y_pred & valid_pixels
                )

                # Release temporary arrays before next window
                del pred_window, gt_window
                del y_pred, y_true, valid_pixels

        # Calculate metrics from accumulated pixel counts
        precision = TP / (TP + FP) if (TP + FP) else 0.0
        recall = TP / (TP + FN) if (TP + FN) else 0.0

        f1_denominator = 2 * TP + FP + FN
        f1 = (
            2 * TP / f1_denominator
            if f1_denominator else 0.0
        )

        iou_denominator = TP + FP + FN
        iou = (
            TP / iou_denominator
            if iou_denominator else 0.0
        )

    print(
        f"\n===== RESULTS: "
        f"{os.path.basename(pred_raster_path)} ====="
    )
    print(f"True Positives  : {TP:,}")
    print(f"False Positives : {FP:,}")
    print(f"False Negatives : {FN:,}")
    print(f"Recall          : {recall:.4f}")
    print("========================================================")

    # Release memory used by vector data and spatial index
    del gdf_gt, spatial_index
    gc.collect()


# ==========================================================
# PATHS
# ==========================================================

GT_PATH = (
    "/content/drive/MyDrive/Image data/"
    "Ground truth digitized data/Diepsloot ground truth.gpkg"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/Image data/"
    "Diepsloot_Generalization_Results"
)


# ==========================================================
# EVALUATE ALL THREE MODELS
# ==========================================================

prediction_files = [
    "maskrcnn_diepsloot_prediction.tif"
]

for filename in prediction_files:
    evaluate_model_generalization(
        GT_PATH,
        os.path.join(RESULTS_DIR, filename),
        window_size=512
    )

    gc.collect()



===== RESULTS: maskrcnn_diepsloot_prediction.tif =====
True Positives  : 14,350,367
False Positives : 61,547,045
False Negatives : 18,237,940
Recall          : 0.4404


In [7]:
import os
import numpy as np
import geopandas as gpd
import rasterio
from shapely.geometry import box
from rasterio.windows import Window

GT_PATH = (
    "/content/drive/MyDrive/Image data/"
    "Ground truth digitized data/Diepsloot ground truth.gpkg"
)

RESULTS_DIR = (
    "/content/drive/MyDrive/Image data/"
    "Diepsloot_Generalization_Results"
)

prediction_files = [
    "maskrcnn_diepsloot_prediction.tif"
]

gdf = gpd.read_file(GT_PATH)

print("GROUND TRUTH")
print("CRS:", gdf.crs)
print("Number of features:", len(gdf))
print("Ground-truth bounds:", gdf.total_bounds)
print("Invalid geometries:", (~gdf.geometry.is_valid).sum())

for filename in prediction_files:
    path = os.path.join(RESULTS_DIR, filename)

    print(f"\n{'=' * 60}")
    print("FILE:", filename)

    if not os.path.exists(path):
        print("FILE NOT FOUND")
        continue

    with rasterio.open(path) as src:
        print("CRS:", src.crs)
        print("Dimensions:", src.width, "x", src.height)
        print("Resolution:", src.res)
        print("Bounds:", src.bounds)
        print("NoData value:", src.nodata)

        # Compare geographic coverage
        gt_aligned = gdf.to_crs(src.crs)
        raster_extent = box(*src.bounds)
        gt_extent = box(*gt_aligned.total_bounds)

        print(
            "Ground truth overlaps raster:",
            raster_extent.intersects(gt_extent)
        )
        print(
            "Ground truth extent fully within raster:",
            raster_extent.covers(gt_extent)
        )

        # Inspect only a small sample using an explicit Window object
        sample = src.read(
            1,
            window=Window(
                0, 0,
                min(512, src.width),
                min(512, src.height)
            )
        )

        print("Sample unique values:", np.unique(sample)[:20])
        print("Sample minimum:", sample.min())
        print("Sample maximum:", sample.max())

GROUND TRUTH
CRS: EPSG:3857
Number of features: 377
Ground-truth bounds: [ 3118027.20334199 -2991429.3785582   3118257.97671426 -2991199.89672869]
Invalid geometries: 0

FILE: maskrcnn_diepsloot_prediction.tif
CRS: EPSG:3857
Dimensions: 28087 x 19866
Resolution: (0.023063499127712717, 0.023063500453039648)
Bounds: BoundingBox(left=3117921.2035, bottom=-2991622.8352, right=3118568.988, top=-2991164.6557)
NoData value: None
Ground truth overlaps raster: True
Ground truth extent fully within raster: True
Sample unique values: [0]
Sample minimum: 0
Sample maximum: 0
